### 2. DATA INGESTION: DOWNLOAD FROM KAGGLE AND LOAD TO DATABASE


In [ ]:
import gc
import io
import os
import sys
import zipfile
from datetime import datetime
from pathlib import Path

# Tells Python to look one folder up (the root folder)
sys.path.append(os.path.abspath(os.path.join("..")))

from kaggle.api.kaggle_api_extended import KaggleApi

from db_connect import connect_to_db

# 1. Setup local environment target file path
raw_data_dir = "../data/01-raw/"

# Ensure the local raw storage folder exists
os.makedirs(raw_data_dir, exist_ok=True)

# DIAGNOSTIC: Check what files/folders already exist in raw_data_dir
print("🔎 Checking existing files in data/01-raw/...")
existing_items = os.listdir(raw_data_dir)
print(f"   Found: {existing_items}\n")

# Look for CSV files directly in raw_data_dir (NOT in subdirectories)
csv_files_direct = list(Path(raw_data_dir).glob("*.csv"))
print(f"   CSV files in raw_data_dir: {[f.name for f in csv_files_direct]}\n")

# Determine which CSV to use - prioritize the main one, not the copy
if csv_files_direct:
    # Sort to get the main file first (without "Copy" in name)
    main_csv = [f for f in csv_files_direct if "Copy" not in f.name]
    if main_csv:
        csv_file = main_csv[0]
    else:
        csv_file = csv_files_direct[0]
    print(f"✓ Using CSV file: {csv_file.name}\n")
else:
    print("❌ No CSV files found in data/01-raw/")
    csv_file = None

# 2. Optimized Incremental Daily Update Pipeline
if csv_file:
    conn = connect_to_db()

    if conn:
        print("📊 Starting incremental daily update pipeline...")

        # INSPECT CSV COLUMNS FIRST
        print("\n📋 Inspecting CSV columns...")
        sample_chunk = pd.read_csv(csv_file, nrows=1)
        print(f"   CSV columns: {list(sample_chunk.columns)}\n")

        # PRE-INGESTION FILTER: Fetch existing composite keys (video_id + trending_date + country)
        print("🔍 Fetching existing records from database...")
        query_existing_records = """
            SELECT DISTINCT video_id, video_trending__date, video_trending_country
            FROM youtube_data_schema.youtube_trending_videos_global;
        """
        try:
            existing_records_df = pd.read_sql_query(query_existing_records, conn)
            # Convert to set of tuples for O(1) lookup - composite key prevents duplicates
            existing_records = set()
            for idx, row in existing_records_df.iterrows():
                try:
                    # Standardize date format - handle ISO 8601 dates
                    trending_date = pd.to_datetime(
                        row["video_trending__date"], format="ISO8601"
                    ).strftime("%Y-%m-%d")
                    composite_key = (row["video_id"], trending_date, row["video_trending_country"])
                    existing_records.add(composite_key)
                except:
                    pass

            print(f"✓ Found {len(existing_records)} existing records in database")
            if len(existing_records) > 0:
                print(
                    f"  (Identified by: video_id + video_trending__date + video_trending_country)"
                )
        except Exception as e:
            print(f"⚠️  Could not fetch existing records (table may be empty): {e}")
            existing_records = set()

        # CHUNKED FILTERING LOOP: Process CSV with deduplication
        print("\n🔄 Streaming dataset into database with incremental filtering...")
        print(f"Processing: {csv_file.name}\n")

        try:
            # Read in chunks to handle large files
            total_rows_read = 0
            total_rows_inserted = 0
            total_rows_filtered = 0
            chunks = pd.read_csv(csv_file, chunksize=5000)

            # Get cursor for raw SQL inserts
            cursor = conn.cursor()

            for chunk_idx, chunk in enumerate(chunks, 1):
                total_rows_read += len(chunk)

                # Create composite keys for deduplication
                if (
                    "video_id" in chunk.columns
                    and "video_trending__date" in chunk.columns
                    and "video_trending_country" in chunk.columns
                ):
                    # Standardize date format - use ISO8601 format since CSV has that format
                    chunk["video_trending__date"] = pd.to_datetime(
                        chunk["video_trending__date"], format="ISO8601", utc=True
                    ).dt.strftime("%Y-%m-%d")

                    # Create composite keys for each row
                    chunk["_composite_key"] = chunk.apply(
                        lambda row: (
                            row["video_id"],
                            row["video_trending__date"],
                            row["video_trending_country"],
                        ),
                        axis=1,
                    )

                    # Filter: Keep only rows where composite_key is NOT in existing_records
                    mask_new = ~chunk["_composite_key"].isin(existing_records)
                    chunk_filtered = chunk[mask_new].drop(columns=["_composite_key"]).copy()

                    rows_filtered_this_chunk = len(chunk) - len(chunk_filtered)
                    total_rows_filtered += rows_filtered_this_chunk
                else:
                    # If required columns don't exist, show what we have
                    available_cols = list(chunk.columns)
                    print(f"⚠️  Required columns not found in CSV!")
                    print(f"    Available columns: {available_cols}")
                    chunk_filtered = chunk.copy()

                # Insert only the new rows using raw INSERT statements
                if len(chunk_filtered) > 0:
                    try:
                        # Build column list and INSERT statement
                        columns = ", ".join(chunk_filtered.columns)
                        placeholders = ", ".join(["%s"] * len(chunk_filtered.columns))
                        insert_sql = f"INSERT INTO youtube_data_schema.youtube_trending_videos_global ({columns}) VALUES ({placeholders})"

                        # Execute batch insert
                        rows_inserted_this_chunk = 0
                        for idx, row in chunk_filtered.iterrows():
                            try:
                                cursor.execute(insert_sql, tuple(row))
                                rows_inserted_this_chunk += 1
                            except Exception as row_err:
                                # Silently skip rows that fail (likely constraint violations)
                                pass

                        # Commit after every chunk
                        conn.commit()
                        total_rows_inserted += rows_inserted_this_chunk

                        # Update the in-memory set with newly inserted composite keys
                        if (
                            "video_id" in chunk_filtered.columns
                            and "video_trending__date" in chunk_filtered.columns
                        ):
                            for idx, row in chunk_filtered.iterrows():
                                new_key = (
                                    row["video_id"],
                                    row["video_trending__date"],
                                    row["video_trending_country"],
                                )
                                existing_records.add(new_key)

                        if rows_inserted_this_chunk > 0:
                            print(
                                f"  [Chunk {chunk_idx}] Read: {len(chunk)} | New: {len(chunk_filtered)} | Inserted: {rows_inserted_this_chunk} | Duplicates skipped: {rows_filtered_this_chunk} | Total inserted: {total_rows_inserted}"
                            )
                        else:
                            print(
                                f"  [Chunk {chunk_idx}] Read: {len(chunk)} | New rows filtered: {len(chunk_filtered)} | Duplicates skipped: {rows_filtered_this_chunk}"
                            )
                    except Exception as insert_err:
                        print(f"  ⚠️  Chunk {chunk_idx} insert error: {insert_err}")
                        conn.rollback()  # Rollback failed transaction
                else:
                    print(
                        f"  [Chunk {chunk_idx}] Read: {len(chunk)} | All rows already exist (skipped)"
                    )

                # Force garbage collection after each chunk
                gc.collect()

            cursor.close()
            print(f"\n✅ Incremental update complete!")
            print(f"   📊 Total rows read from CSV: {total_rows_read}")
            print(f"   ✨ New rows inserted: {total_rows_inserted}")
            print(f"   ⏭️  Duplicate rows skipped: {total_rows_filtered}")

        except Exception as e:
            print(f"❌ Error processing {csv_file.name}: {e}")
            import traceback

            traceback.print_exc()

        # Verify final count
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM youtube_data_schema.youtube_trending_videos_global;")
        final_count = cursor.fetchone()[0]
        cursor.close()
        print(f"\n📊 Final database record count: {final_count}")

        conn.close()
    else:
        print("❌ Could not connect to database.")
else:
    print("❌ Cannot proceed without CSV file")